# Walkthrough — Few-Step Generation, visualized

A visual tour of the project. It shows, for **Flow Matching**:

1. Samples as the number of sampling steps grows (1 → 16).
2. The **sampling trajectory**: how each image forms, noise → digit, step by step.
3. The effect of classifier-free guidance.
4. An optional DDPM comparison.

Runs both **locally** (from the repo root) and on **Kaggle** (clones the repo). If a
trained checkpoint exists under `results/`, it is loaded; otherwise a small model is
trained on the spot. For the full FID-vs-steps study see `02_results.ipynb`.

In [ ]:
# Setup: import the package locally, or clone it on Kaggle.
import os, sys, subprocess
try:
    import fmfs  # noqa: F401
except ModuleNotFoundError:
    if not os.path.isdir('flow-matching-fewstep'):
        subprocess.run(['git', 'clone', '-q', 'https://github.com/xHansi/flow-matching-fewstep.git'])
        subprocess.run(['pip', 'install', '-q', 'pytorch-fid'])
    sys.path.insert(0, os.path.abspath('flow-matching-fewstep'))
    import fmfs  # noqa: F401

In [ ]:
import torch
from pathlib import Path
from IPython.display import Image, display
from fmfs.models import UNet
from fmfs.flow import make_method
from fmfs.data import make_loaders
from fmfs.inference import load_checkpoint
from fmfs.utils import EMA, get_device, save_samples, save_trajectory, set_seed

device = get_device(); set_seed(0)
os.makedirs('demo_out', exist_ok=True)
print('device:', device)

def show(imgs, name, nrow=10):
    save_samples(imgs, f'demo_out/{name}.png', nrow=nrow); display(Image(f'demo_out/{name}.png'))

def show_traj(traj, name):
    save_trajectory(traj, f'demo_out/{name}.png'); display(Image(f'demo_out/{name}.png'))

## Load or quickly train a Flow Matching model

If `results/flow_mnist/ckpt.pt` exists it is loaded (best quality). Otherwise a small
model is trained for `MAX_STEPS` — enough to *see* the few-step behaviour. Raise it (or
train via `python -m scripts.train --method flow --epochs 30`) for sharper digits.

In [ ]:
arch = dict(in_ch=1, base=64, num_classes=10, image_size=32)
CKPT = Path('results/flow_mnist/ckpt.pt')
MAX_STEPS = 2000  # only used if no checkpoint is found

if CKPT.exists():
    model, flow, _ = load_checkpoint(str(CKPT), device)
    print('loaded checkpoint:', CKPT)
else:
    from torch import optim
    from tqdm.auto import tqdm
    model = UNet(**arch).to(device); flow = make_method('flow')
    ema = EMA(model); opt = optim.AdamW(model.parameters(), 2e-4)
    loader, _ = make_loaders('mnist', batch_size=128, num_workers=2)
    step = 0; pbar = tqdm(total=MAX_STEPS)
    while step < MAX_STEPS:
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            yd = y.clone(); yd[torch.rand(y.size(0), device=device) < 0.1] = model.null_class
            loss = flow.loss(model, x, yd)
            opt.zero_grad(); loss.backward(); opt.step(); ema.update(model)
            step += 1; pbar.update(1); pbar.set_postfix(loss=float(loss))
            if step >= MAX_STEPS: break
    m = UNet(**arch).to(device); ema.copy_to(m); model = m
    print('trained', MAX_STEPS, 'steps')
model.eval();

## 1. Samples vs. number of steps

Each column is a digit class. Even at **1–2 steps** Flow Matching already produces
recognizable digits; a handful of steps is enough for clean samples.

In [ ]:
y = torch.arange(10, device=device).repeat(10)
for s in (1, 2, 4, 8, 16):
    print(f'Flow Matching — {s} sampling step(s)')
    show(flow.sample(model, y, steps=s, cfg_scale=2.0), f'flow_{s}steps')

## 2. The sampling trajectory (noise → digit)

Each **row** is one digit; columns go left-to-right from pure noise to the final image.
This is the ODE being integrated in a few Euler steps.

In [ ]:
y = torch.arange(10, device=device)
_, traj8 = flow.sample(model, y, steps=8, cfg_scale=2.0, return_trajectory=True)
print('8-step trajectory (9 columns: noise -> digit)'); show_traj(traj8, 'flow_traj_8')
_, traj16 = flow.sample(model, y, steps=16, cfg_scale=2.0, return_trajectory=True)
print('16-step trajectory (finer)'); show_traj(traj16, 'flow_traj_16')

## 3. Effect of classifier-free guidance

Higher guidance sharpens class identity but can over-saturate. FID in `02_results.ipynb`
is therefore measured **without** guidance (`cfg=1.0`).

In [ ]:
y = torch.arange(10, device=device).repeat(5)
for w in (1.0, 2.0, 3.0):
    print(f'Flow Matching — 8 steps, guidance = {w}')
    show(flow.sample(model, y, steps=8, cfg_scale=w), f'flow_cfg{w}', nrow=10)

## 4. Optional: DDPM comparison

If a DDPM checkpoint exists, compare the same few-step regime. DDPM (DDIM, `eta=0`)
needs more steps than Flow Matching to look clean.

In [ ]:
DCKPT = Path('results/ddpm_mnist/ckpt.pt')
if DCKPT.exists():
    dmodel, ddpm, _ = load_checkpoint(str(DCKPT), device)
    y = torch.arange(10, device=device).repeat(10)
    for s in (2, 8, 50):
        print(f'DDPM (DDIM) — {s} steps'); show(ddpm.sample(dmodel, y, steps=s, cfg_scale=1.0, eta=0.0), f'ddpm_{s}steps')
    _, dtraj = ddpm.sample(dmodel, torch.arange(10, device=device), steps=16, cfg_scale=1.0, return_trajectory=True)
    print('DDPM trajectory (16 steps)'); show_traj(dtraj, 'ddpm_traj_16')
else:
    print('No DDPM checkpoint. Train one with:  python -m scripts.train --method ddpm --epochs 30')

---
For the quantitative story (FID vs. steps for both methods and reflow, plus the two FID
metrics) run **`02_results.ipynb`**.